In [ ]:
%matplotlib widget

import ipywidgets as widgets
from ipywidgets import HBox, VBox, Box, Layout, Output, Label
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import display, Latex, Markdown, Image, Math

In [ ]:
def make_box_layout():
     return widgets.Layout(
        border='solid 1px black',
        margin='0px 5px 5px 0px',
        padding='2px 2px 2px 2px'
     )

## Störgrößenkompensation mittels Regulator

In [ ]:
out = Output()

with out:
    display(Markdown(r"""
- Gleichstrommotor mit Lastmoment $\tau_\text{L}(t)$, Eingang Ankerspannung $u(t)$, Ausgang Drehgeschwindigkeit $y(t) = \omega(t)$
  \begin{align*}
      L_\text{A} \dot{i}(t) & = -R\, i(t) - k\, \omega(t) + u(t), &
      J \dot{\omega}(t) & = k\, i(t) - \nu\, \omega(t) - \tau_\text{L}(t)
  \end{align*}
- Lastmoment aus konstantem und harmonischem Anteil $\tau_\text{L}(t) = d_0 + d_1 \sin(\omega_\text{L} t)$, beschrieben durch das Exosystem
  \begin{align*}
      \dot{\boldsymbol{z}}(t) & = A_\text{ex} \boldsymbol{z}(t) = \begin{pmatrix} 0 & 0 & 0 \\ 0 & 0 & \omega_\text{L} \\ 0 & -\omega_\text{L} & 0 \end{pmatrix} \boldsymbol{z}(t), &
      \tau_\text{L}(t) & = C_\text{ex} \boldsymbol{z}(t) = \begin{pmatrix} 1 & 1 & 0 \end{pmatrix} \boldsymbol{z}(t)
  \end{align*}
- Das Lastmoment greift nicht am Eingang an, $P = (0, -\frac{1}{J})^\intercal$, $Q = 0$
- Regelgesetz $u(t) = u_\text{ref}(t) - \boldsymbol{k}^\intercal (\boldsymbol{x}(t) - \boldsymbol{x}_\text{ref}(t)) + \boldsymbol{l}^\intercal \boldsymbol{z}(t)$
  mit $\boldsymbol{l}^\intercal = \boldsymbol{\gamma}^\intercal + \boldsymbol{k}^\intercal \Pi$ aus den Regulatorgleichungen
  \begin{align*}
      A \Pi + \boldsymbol{b} \boldsymbol{\gamma}^\intercal + P C_\text{ex} & = \Pi A_\text{ex}, &
      \boldsymbol{c}^\intercal \Pi + Q C_\text{ex} & = \boldsymbol{0}
  \end{align*}
- Vergleich
    - Zustandsregler mit Steuerung
    - Zustandsregler mit Steuerung und Regulator (Störung bekannt)
    - Zustandsregler mit Steuerung und Regulator, Zustand und Störung aus Beobachter
    """))
display(out)

__Definition Parameter__

In [ ]:
R = 1.0       # Ankerwiderstand in Ohm
LA = 0.1      # Ankerinduktivität in H
k = 0.5       # Motorkonstante in Nm/A
J = 0.05      # Trägheitsmoment in kg m^2
nu = 0.01     # Dämpfung in Nm s

In [ ]:
tSim = np.linspace(0, 6, 1201)

## Systemmatrizen

In [ ]:
A = np.array([[-R / LA, -k / LA],
              [k / J, -nu / J]])
b = np.array([1 / LA, 0])
c = np.array([0, 1])
P = np.array([[0], [-1 / J]])
Q = np.array([[0]])

In [ ]:
def exosystem(wL):
    Aex = np.array([[0, 0, 0],
                    [0, 0, wL],
                    [0, -wL, 0]])
    Cex = np.array([[1, 1, 0]])
    return Aex, Cex

## Entwurf von Zustandsregler, Regulator und Beobachter

In [ ]:
def acker(A, b, poles):
    """Ackermannformel für ein System mit einem Eingang"""
    n = A.shape[0]
    S = np.column_stack([np.linalg.matrix_power(A, i) @ b for i in range(n)])
    coeff = np.real(np.poly(poles))
    pA = sum(ci * np.linalg.matrix_power(A, n - i) for i, ci in enumerate(coeff))
    q = np.linalg.solve(S.T, np.eye(n)[:, -1])
    return q @ pA

In [ ]:
def regulator(A, b, c, P, Q, Aex, Cex):
    """Lösung der Regulatorgleichungen
        A Pi + b gamma^T + P Cex = Pi Aex
        c^T Pi + Q Cex = 0
    durch Vektorisierung mit dem Kronecker-Produkt"""
    n = A.shape[0]
    nz = Aex.shape[0]
    B = b.reshape(-1, 1)
    C = c.reshape(1, -1)
    M = np.block([[np.kron(np.eye(nz), A) - np.kron(Aex.T, np.eye(n)), np.kron(np.eye(nz), B)],
                  [np.kron(np.eye(nz), C), np.zeros((nz, nz))]])
    rhs = -np.concatenate([(P @ Cex).flatten(order='F'), (Q @ Cex).flatten(order='F')])
    sol = np.linalg.solve(M, rhs)
    Pi = sol[:n * nz].reshape((n, nz), order='F')
    gamma = sol[n * nz:]
    return Pi, gamma

In [ ]:
def printMatrix(M):
    M = np.atleast_2d(M)
    rows = [r" & ".join(f"{v:.4g}" for v in row) for row in M]
    return r"\begin{pmatrix}" + r" \\ ".join(rows) + r"\end{pmatrix}"

In [ ]:
Aex, Cex = exosystem(2 * np.pi)
Pi, gamma = regulator(A, b, c, P, Q, Aex, Cex)
display(Math(r"\Pi = " + printMatrix(Pi) + r",\qquad \boldsymbol{\gamma}^\intercal = " + printMatrix(gamma)))

Die erste Zeile von $\Pi$ zeigt, dass der Strom im stationären Regime um $\frac{\tau_\text{L}}{k}$ ausgelenkt werden muss, damit die Drehgeschwindigkeit trotz Lastmoment der Referenz folgt.

## Simulation geschlossener Kreis

In [ ]:
def trajectory(t, wd, t0, dT):
    """Polynomiale Überführung phi_2 der Drehgeschwindigkeit und deren Ableitungen"""
    tau = np.clip((t - t0) / dT, 0, 1)
    eta = wd * (10 * tau ** 3 - 15 * tau ** 4 + 6 * tau ** 5)
    if 0 < tau < 1:
        deta = wd / dT * (30 * tau ** 2 - 60 * tau ** 3 + 30 * tau ** 4)
        ddeta = wd / dT ** 2 * (60 * tau - 180 * tau ** 2 + 120 * tau ** 3)
    else:
        deta, ddeta = 0, 0
    return eta, deta, ddeta

def feedforward(t, wd, t0, dT):
    """Flachheitsbasierte Steuerung mit flachem Ausgang omega"""
    w, dw, ddw = trajectory(t, wd, t0, dT)
    i = (J * dw + nu * w) / k
    di = (J * ddw + nu * dw) / k
    u = LA * di + R * i + k * w
    return np.array([i, w]), u

In [ ]:
def disturbance(t, d0, d1, wL, td0, td1):
    """Zustand des Exosystems: konstanter Anteil ab td0, harmonischer Anteil ab td1"""
    return np.array([d0 if t >= td0 else 0,
                     d1 * np.sin(wL * (t - td1)) if t >= td1 else 0,
                     d1 * np.cos(wL * (t - td1)) if t >= td1 else 0])

In [ ]:
def closedLoop(t, s, mode, K, l, lObs, At, bt, ct, Cex, ffArgs, distArgs):
    x, r = s[:2], s[2:]
    xRef, uRef = feedforward(t, *ffArgs)
    z = disturbance(t, *distArgs)

    if mode == 'Zustandsregler':
        u = uRef - K @ (x - xRef)
    elif mode == 'Zustandsregler mit Regulator':
        u = uRef - K @ (x - xRef) + l @ z
    else:
        u = uRef - K @ (r[:2] - xRef) + l @ r[2:]

    y = c @ x
    dx = A @ x + b * u + P[:, 0] * (Cex @ z)[0]
    dr = At @ r + bt * u + lObs * (y - ct @ r)

    return np.concatenate([dx, dr])

In [ ]:
output = widgets.Output()

with output:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.subplots_adjust(bottom=0.08, top=0.93, left=0.1, right=0.9, hspace=0.1)

ax1.set_xlim([0, tSim[-1]])
ax1.grid()
ax1.set_ylabel(r"$\omega$ in rad/s")
ax11 = ax1.twinx()
ax11.set_ylabel(r'$u$ in V', color='C1')
ax11.tick_params(axis='y', colors='C1')
ax2.grid()
ax2.set_xlabel(r"$t$ in s")
ax2.set_ylabel(r"$\tau_\mathrm{L}$ in Nm")

wRefSys, = ax1.plot([], [], 'C0', label=r'Referenz')
wSys, = ax1.plot([], [], 'C2', label=r'System')
uSys, = ax11.plot([], [], 'C1:', label=r'Stellgröße')
dSys, = ax2.plot([], [], 'C3', label=r'Lastmoment')
dObs, = ax2.plot([], [], 'C4--', label=r'Lastmoment (Beobachter)')

handlesAx, labelsAx = [], []
for ax in [ax1, ax11, ax2]:
    h, la = ax.get_legend_handles_labels()
    handlesAx += h
    labelsAx += la

fig.legend(handlesAx, labelsAx,
           bbox_to_anchor=(0.1, 0.94, 0.8, .15), loc=3,
           ncol=5, mode="expand", borderaxespad=0., framealpha=0.5)

sliderWd = widgets.FloatSlider(value=10,
                               min=1,
                               max=30,
                               step=1,
                               description=r"$\omega_\text{d}$")
sliderD0 = widgets.FloatSlider(value=0.2,
                               min=0,
                               max=0.5,
                               step=0.05,
                               description=r"$d_0$")
sliderD1 = widgets.FloatSlider(value=0.1,
                               min=0,
                               max=0.5,
                               step=0.05,
                               description=r"$d_1$")
sliderF = widgets.FloatSlider(value=1,
                              min=0.2,
                              max=5,
                              step=0.1,
                              description=r"$f_\text{L}$")
sliderPoleK = widgets.FloatSlider(value=-20,
                                  min=-50,
                                  max=-2,
                                  step=1,
                                  description=r"$s_{\text{R}}$")
sliderPoleL = widgets.FloatSlider(value=-30,
                                  min=-80,
                                  max=-5,
                                  step=1,
                                  description=r"$s_{\text{B}}$")

radioU = widgets.RadioButtons(options=['Regler', 'Regler mit Regulator', 'Regler mit Regulator und Beobachter'],
                              disabled=False)

def calcSystem(_):
    wd = sliderWd.value
    d0 = sliderD0.value
    d1 = sliderD1.value
    wL = 2 * np.pi * sliderF.value
    mode = radioU.value
    ffArgs = (wd, 0.5, 0.5)
    distArgs = (d0, d1, wL, 1.5, 3.0)

    Aex, Cex = exosystem(wL)
    K = acker(A, b, [sliderPoleK.value] * 2)
    Pi, gamma = regulator(A, b, c, P, Q, Aex, Cex)
    l = gamma + K @ Pi

    At = np.block([[A, P @ Cex], [np.zeros((3, 2)), Aex]])
    bt = np.concatenate([b, np.zeros(3)])
    ct = np.concatenate([c, (Q @ Cex)[0]])
    lObs = acker(At.T, ct, [sliderPoleL.value] * 5)

    res = solve_ivp(closedLoop,
                    [tSim[0], tSim[-1]],
                    np.zeros(7),
                    t_eval=tSim,
                    max_step=2e-3,
                    args=(mode, K, l, lObs, At, bt, ct, Cex, ffArgs, distArgs))

    wRef = np.array([trajectory(t, *ffArgs)[0] for t in res.t])
    uAll = np.zeros(len(res.t))
    for idx, t in enumerate(res.t):
        xRef, uRef = feedforward(t, *ffArgs)
        x, r = res.y[:2, idx], res.y[2:, idx]
        if mode == 'Regler':
            uAll[idx] = uRef - K @ (x - xRef)
        elif mode == 'Regler mit Regulator':
            uAll[idx] = uRef - K @ (x - xRef) + l @ disturbance(t, *distArgs)
        else:
            uAll[idx] = uRef - K @ (r[:2] - xRef) + l @ r[2:]
    dAll = np.array([(Cex @ disturbance(t, *distArgs))[0] for t in res.t])
    dHat = (Cex @ res.y[4:, :])[0]

    wRefSys.set_data(res.t, wRef)
    wSys.set_data(res.t, res.y[1])
    uSys.set_data(res.t, uAll)
    dSys.set_data(res.t, dAll)
    if mode == 'Regler mit Regulator und Beobachter':
        dObs.set_data(res.t, dHat)
    else:
        dObs.set_data([], [])

    ax1.set_ylim([np.min(res.y[1]) - 1, np.max(res.y[1]) + 2])
    ax11.set_ylim([np.min(uAll) - 1, np.max(uAll) + 2])
    ax2.set_ylim([np.min(dAll) - 0.1, np.max(dAll) + 0.1])

    fig.canvas.draw()

sliderWd.observe(calcSystem, names='value')
sliderD0.observe(calcSystem, names='value')
sliderD1.observe(calcSystem, names='value')
sliderF.observe(calcSystem, names='value')
sliderPoleK.observe(calcSystem, names='value')
sliderPoleL.observe(calcSystem, names='value')
radioU.observe(calcSystem, names='value')

calcSystem(None)

bDisturbance = VBox([Label(value='Lastmoment'), sliderD0, sliderD1, sliderF])
bDisturbance.layout = make_box_layout()
bFeedForward = VBox([Label(value='Steuerung'), sliderWd])
bFeedBackType = VBox([Label(value='Regler'), radioU])
bFeedBackParams = VBox([Label(value='Eigenwerte Regler/Beobachter'), sliderPoleK, sliderPoleL])
bFeedBack = HBox([VBox([bFeedForward, bFeedBackType]), bFeedBackParams])
bFeedBack.layout = make_box_layout()

controls = HBox([bDisturbance, bFeedBack])
VBox([controls, output], layout=Layout(display='flex', flex_flow='column', justify_content='center', align_items='center'))